In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes huggingface_hub torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 146.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.5 MB/s eta 0:00:00


In [ ]:
# 1. প্রয়োজনীয় লাইব্রেরি ইনস্টল করা
# !pip install -q -U transformers datasets peft accelerate bitsandbytes huggingface_hub torchao

import torch
import gc
import random
import re
import os
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from PIL import Image
from huggingface_hub import login
from google.colab import userdata

# মডেল লোড করার আগে GPU মেমোরি পুরোপুরি ক্লিয়ার করা
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------
# 2. Hugging Face Login
# ---------------------------------------------------------
try:
    hf_token = userdata.get('hf-rw')
    login(token=hf_token)
    print("✅ Logged in to Hugging Face successfully via Colab Secrets!")
except Exception as e:
    print("❌ Warning: Could not load Colab Secrets. Please add your token manually.")

# ---------------------------------------------------------
# 3. Model & Processor Setup (4-bit QLoRA Optimized)
# ---------------------------------------------------------
model_id = "google/medgemma-4b-it"
print("Loading Processor and Base Model in 4-bit...")

processor = AutoProcessor.from_pretrained(model_id, token=hf_token)

# ✅ 4-bit কোয়ান্টাইজেশন আবার যোগ করা হলো
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # L4 GPU এর জন্য compute_dtype BF16 রাখা হলো
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=hf_token
)

# ✅ 4-bit ট্রেইনিংয়ের জন্য মডেল প্রস্তুত করা হলো
model = prepare_model_for_kbit_training(model)

# ⚠️ LoRA r=32 করা হলো (মেমোরি এবং পারফরম্যান্সের সেরা ব্যালেন্সের জন্য)
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False

# ---------------------------------------------------------
# 4. Dataset Loading & ECG-LM Optimized Augmentation
# ---------------------------------------------------------
print("Loading Dataset...")
dataset = load_dataset("kazi420/ECG_LM_Final_Dataset_V3")

LEADS = ['lead I', 'lead II', 'lead III', 'lead aVR', 'lead aVL', 'lead aVF',
         'lead V1', 'lead V2', 'lead V3', 'lead V4', 'lead V5', 'lead V6']
SEARCH_PATTERNS = {lead: re.compile(rf"{re.escape(lead)}:\s*([^,.\n]+)") for lead in LEADS}

def extract_leads_from_text(text_sequence):
    lead_descriptions = []
    for lead in LEADS:
        match = SEARCH_PATTERNS[lead].search(text_sequence)
        if match:
            desc = match.group(1).strip()
            lead_descriptions.append(f"{lead}: {desc}")
        else:
            lead_descriptions.append(f"{lead}: normal morphology")
    return ", ".join(lead_descriptions)

def apply_lead_masking_optimized(text_sequence, mask_prob=0.94):
    if random.random() > mask_prob:
        return extract_leads_from_text(text_sequence), extract_leads_from_text(text_sequence)

    k = random.randint(4, 6)
    leads_with_abnormality = []
    leads_normal = []
    lead_descriptions = {}

    for lead in LEADS:
        match = SEARCH_PATTERNS[lead].search(text_sequence)
        if match:
            description = match.group(1).strip()
            lead_descriptions[lead] = description
            if description.lower() == "normal morphology":
                leads_normal.append(lead)
            else:
                leads_with_abnormality.append(lead)
        else:
            lead_descriptions[lead] = "normal morphology"
            leads_normal.append(lead)

    max_abnormality_to_drop = max(0, len(leads_with_abnormality) - 1)
    abnormality_drop_count = min(k, max_abnormality_to_drop)
    normal_drop_count = min(k - abnormality_drop_count, len(leads_normal))

    leads_to_drop = []
    if abnormality_drop_count > 0:
        leads_to_drop.extend(random.sample(leads_with_abnormality, abnormality_drop_count))
    if normal_drop_count > 0:
        leads_to_drop.extend(random.sample(leads_normal, normal_drop_count))

    masked_leads = []
    full_leads = []
    for lead in LEADS:
        desc = lead_descriptions.get(lead, "normal morphology")
        full_leads.append(f"{lead}: {desc}")
        if lead not in leads_to_drop:
            masked_leads.append(f"{lead}: {desc}")
        else:
            masked_leads.append(f"{lead}: ")

    return ", ".join(masked_leads), ", ".join(full_leads)

def parse_ecg_text(text_sequence):
    text_sequence = text_sequence.replace("<ECG_IMAGE_TOKEN>", "").strip()
    if "Findings:" in text_sequence:
        patient_profile, rest = text_sequence.split("Findings:", 1)
        patient_profile = patient_profile.strip()
        findings_match = re.search(r"Findings:\s*([^.]+\.)", text_sequence)
        diagnosis_match = re.search(r"Diagnosis:\s*([^.]+\.)", text_sequence)
        findings_text = findings_match.group(1).strip() if findings_match else ""
        diagnosis_text = diagnosis_match.group(1).strip() if diagnosis_match else ""
        ecg_report = f"{findings_text} {diagnosis_text}".strip()
    else:
        patient_profile = text_sequence.split(".")[0]
        ecg_report = text_sequence

    rhythm_match = re.search(r"rhythm \((\d+)\s*bpm\)", text_sequence)
    rhythm_info = rhythm_match.group(0) if rhythm_match else ""
    return patient_profile, ecg_report, rhythm_info

def collate_fn(examples):
    texts = []
    images = []
    for ex in examples:
        original_seq = ex["text_sequence"]
        patient_profile, ecg_report, rhythm_info = parse_ecg_text(original_seq)
        masked_leads, full_leads = apply_lead_masking_optimized(original_seq)

        user_prompt = (
            f"{patient_profile}. "
            f"Analyze the ECG image and generate a complete clinical report. "
            f"Some leads are masked. Use visual evidence to complete them.\n\n"
            f"Visible Leads: {masked_leads}"
        )
        assistant_target = f"ECG Report: {ecg_report}. Lead descriptions: {full_leads}"

        messages = [
            {"role": "user", "content": [{"type": "text", "text": user_prompt}, {"type": "image"}]},
            {"role": "assistant", "content": [{"type": "text", "text": assistant_target}]}
        ]

        formatted_text = processor.apply_chat_template(messages, tokenize=False)
        texts.append(formatted_text)

        if ex.get("ecg_image") is not None:
            img = ex["ecg_image"].convert("RGB").resize((512, 512))
            images.append([img])
        else:
            images.append([Image.new('RGB', (512, 512), (0, 0, 0))])

    batch = processor(
        text=texts, images=images, return_tensors="pt", padding=True, max_length=768, truncation=True
    )

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")
    if image_token_id is not None:
        labels[labels == image_token_id] = -100

    for i, formatted_text in enumerate(texts):
        assistant_start = formatted_text.find("assistant")
        if assistant_start != -1:
            user_part = formatted_text[:assistant_start]
            user_tokens = processor.tokenizer(user_part, add_special_tokens=False, return_tensors="pt")["input_ids"]
            user_token_len = user_tokens.shape[1]
            labels[i, :user_token_len] = -100

    batch["labels"] = labels
    return batch

# ---------------------------------------------------------
# 5. Training Setup (Optimized with 4-bit for Safety)
# ---------------------------------------------------------
output_dir = "./medgemma-ecg-final"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    learning_rate=2e-4,
    num_train_epochs=5,

    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    bf16=True,
    remove_unused_columns=False,

    optim="paged_adamw_8bit",
    disable_tqdm=False,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn
)

# ---------------------------------------------------------
# 6. Start Training
# ---------------------------------------------------------
print("🚀 Starting FULL ECG-LM Training (4-bit QLoRA, 5 Epochs)...")
trainer.train()

# ---------------------------------------------------------
# 7. Auto-Push to Hugging Face After Training
# ---------------------------------------------------------
print("✅ Training Complete! Pushing to Hugging Face Hub...")
hf_repo_id = "bappy2001/MedGemma-ECG-LM-Final_5000_V4"

model.config.use_cache = True
trainer.model.push_to_hub(hf_repo_id, commit_message="Uploaded via Colab Pro Auto-Commit (4-bit QLoRA, Extended LoRA)")
processor.push_to_hub(hf_repo_id, commit_message="Processor uploaded via Colab Pro")

print(f"🎉 All Done! Model is safe and ready at: https://huggingface.co/{hf_repo_id}")

✅ Logged in to Hugging Face successfully via Colab Secrets!
Loading Processor and Base Model in 4-bit...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpxtbbkfn4/tokenizer.json:  24%|##3       | 7.98MB / 33.4MB            

🎉 All Done! Model is safe and ready at: https://huggingface.co/bappy2001/MedGemma-ECG-LM-Final_5000_V4
